# 05 - Final Load Preparation

This is the final step of the ETL pipeline. It loads the cleaned dataset,
runs a validation pass, prints a data quality summary report, and saves
the final dataframe to `data/processed/cleaned_data.csv`.

## 1. Imports and Load Cleaned Data

In [19]:
import os
import pathlib
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

# Output directory
PROCESSED_DIR = pathlib.Path(os.path.join("..", "data", "processed"))
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = PROCESSED_DIR / "cleaned_data.csv"

# Restore cleaned dataframe from notebook 02
%store -r df_clean
df = df_clean.copy()
print(f"df_clean loaded: {df.shape}")

df_clean loaded: (100000, 48)


## 2. Validation Pass

Check that the dataframe meets all quality criteria before writing to disk:
- No remaining null values
- All column names are snake_case
- Numeric columns have numeric dtype
- No duplicate rows

In [20]:
import re

issues = []

# Check 1: null values
total_nulls = df.isnull().sum().sum()
if total_nulls > 0:
    issues.append(f"Null values remaining: {total_nulls}")
else:
    print("[PASS] No null values.")

# Check 2: column name format (snake_case — lowercase letters, digits, underscores only)
snake_re = re.compile(r"^[a-z0-9_]+$")
bad_cols  = [c for c in df.columns if not snake_re.match(c)]
if bad_cols:
    issues.append(f"Columns with non-snake_case names: {bad_cols}")
else:
    print("[PASS] All column names are snake_case.")

# Check 3: object columns that should be numeric (heuristic: >90% parseable as float)
obj_cols = df.select_dtypes(include="object").columns
implicitly_numeric = []
for col in obj_cols:
    converted = pd.to_numeric(df[col], errors="coerce")
    if converted.notna().mean() > 0.90:
        implicitly_numeric.append(col)
if implicitly_numeric:
    issues.append(f"Columns stored as object but appear numeric: {implicitly_numeric}")
else:
    print("[PASS] All numeric-looking columns have correct dtype.")

# Check 4: duplicate rows
n_dupes = df.duplicated().sum()
if n_dupes > 0:
    issues.append(f"Duplicate rows: {n_dupes}")
    df.drop_duplicates(inplace=True)
    print(f"[FIXED] Dropped {n_dupes} duplicate rows.")
else:
    print("[PASS] No duplicate rows.")

# Report any issues
if issues:
    print("\n--- Validation issues found ---")
    for iss in issues:
        print(f"  [WARN] {iss}")
else:
    print("\nAll validation checks passed.")

[PASS] No null values.
[PASS] All column names are snake_case.
[PASS] All numeric-looking columns have correct dtype.
[PASS] No duplicate rows.

All validation checks passed.


## 3. Data Quality Summary Report

A structured table summarizing the final state of each column in the dataset.

In [21]:
def build_quality_report(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Build a per-column data quality summary table."""
    rows = []
    for col in dataframe.columns:
        s = dataframe[col]
        null_count   = s.isnull().sum()
        null_pct     = round(s.isnull().mean() * 100, 2)
        n_unique     = s.nunique()
        dtype        = str(s.dtype)

        if pd.api.types.is_numeric_dtype(s):
            col_min = round(s.min(), 4)
            col_max = round(s.max(), 4)
            col_mean= round(s.mean(), 4)
        else:
            top_val = s.mode().iloc[0] if not s.mode().empty else None
            col_min = top_val
            col_max = None
            col_mean= None

        rows.append({
            "column":       col,
            "dtype":        dtype,
            "null_count":   null_count,
            "null_pct":     null_pct,
            "unique_values":n_unique,
            "min_or_mode":  col_min,
            "max":          col_max,
            "mean":         col_mean,
        })
    return pd.DataFrame(rows).set_index("column")


quality_report = build_quality_report(df)
print("Data Quality Summary Report:")
quality_report

Data Quality Summary Report:


,dtype,null_count,null_pct,unique_values,min_or_mode,max,mean
column,,,,,,,
student_id,int64,0,0.0,100000,1,100000.00,50000.5000
country,object,0,0.0,50,Bolivia,NaN,NaN
development_level,object,0,0.0,3,Developing,NaN,NaN
poverty_rate_percent,float64,0,0.0,49,5.21,62.28,26.1827
internet_infrastructure_index,float64,0,0.0,50,20.76,94.39,60.2127
average_internet_speed_mbps,float64,0,0.0,50,5.74,237.90,72.6428
age,int64,0,0.0,11,15,25.00,19.9946
gender,object,0,0.0,3,Male,NaN,NaN
urban_rural,object,0,0.0,2,Urban,NaN,NaN


In [22]:
# High-level quality metrics
total_cells      = df.shape[0] * df.shape[1]
total_null_cells = df.isnull().sum().sum()
completeness_pct = round((1 - total_null_cells / total_cells) * 100, 2)

print("=" * 50)
print("HIGH-LEVEL DATA QUALITY METRICS")
print("=" * 50)
print(f"Total rows        : {df.shape[0]:,}")
print(f"Total columns     : {df.shape[1]}")
print(f"Total cells       : {total_cells:,}")
print(f"Null cells        : {total_null_cells}")
print(f"Completeness      : {completeness_pct}%")
print(f"Duplicate rows    : {df.duplicated().sum()}")
print(f"Numeric columns   : {len(df.select_dtypes(include=[np.number]).columns)}")
print(f"Categorical cols  : {len(df.select_dtypes(include='object').columns)}")
print("=" * 50)

HIGH-LEVEL DATA QUALITY METRICS
Total rows        : 100,000
Total columns     : 48
Total cells       : 4,800,000
Null cells        : 0
Completeness      : 100.0%
Duplicate rows    : 0
Numeric columns   : 38
Categorical cols  : 10


## 4. Save Final Dataframe to CSV

In [23]:
# Save to data/processed/cleaned_data.csv
df.to_csv(OUTPUT_PATH, index=False)

# Verify the file was written
file_size_bytes = os.path.getsize(OUTPUT_PATH)
file_size_mb    = round(file_size_bytes / (1024 ** 2), 2)

print("File saved successfully.")
print(f"Path      : {OUTPUT_PATH.resolve()}")
print(f"Shape     : {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"File size : {file_size_mb} MB ({file_size_bytes:,} bytes)")

File saved successfully.
Path      : /Users/soumyatiwari/Desktop/B_G6_Digitomics/data/processed/cleaned_data.csv
Shape     : 100,000 rows x 48 columns
File size : 25.6 MB (26,841,467 bytes)


## 5. Quick Sanity Read-Back

Re-load the saved CSV to confirm it matches what we wrote.

In [24]:
df_verify = pd.read_csv(OUTPUT_PATH)

shapes_match = df_verify.shape == df.shape
print(f"Shape match   : {shapes_match} (written={df.shape}, read_back={df_verify.shape})")
print(f"Null count    : {df_verify.isnull().sum().sum()}")
print("First 5 rows of the saved file:")
df_verify.head()

Shape match   : True (written=(100000, 48), read_back=(100000, 48))
Null count    : 0
First 5 rows of the saved file:


,student_id,country,development_level,poverty_rate_percent,internet_infrastructure_index,average_internet_speed_mbps,age,gender,urban_rural,family_income_level,device_access,internet_access_hours,education_level,field_of_study,academic_motivation,online_learning_hours,social_media_hours,sessions_per_day,average_session_length_minutes,late_night_usage,education_content_hours,short_video_hours,entertainment_content_hours,news_content_hours,likes_given_per_day,comments_written_per_day,posts_created_per_week,late_night_score,brain_rot_index,brain_rot_level,attention_span_minutes,study_hours_per_week,class_attendance_rate,productivity_score,sleep_hours,stress_level,anxiety_score,depression_score,ads_viewed_per_day,ads_clicked_per_week,impulse_purchase_score,digital_spending_per_month,cyberbullying_exposure,adult_content_exposure,digital_addiction_score,wellbeing_index,academic_risk_score,financial_risk_score
0,1,Canada,Developed,14.51,90.50,188.19,15,Female,Urban,High,Both,6.25,School,Law,6.39,6.70,4.75,13.97,56.63,Sometimes,0.94,2.80,1.00,0.55,132.14,16.08,4.34,1,25.07,Medium,56.95,26.25,90.68,8.45,5.85,5.33,6.04,8.43,139.74,13.72,5.39,106.59,0,0,17.14,54.67,28.84,36.33
1,2,Vietnam,Developing,30.70,66.87,45.63,21,Female,Urban,Middle,Smartphone,6.90,Graduate,Social Sciences,8.00,7.36,4.27,11.72,53.36,Often,0.92,1.42,1.94,0.50,94.87,4.06,7.08,2,25.95,Medium,56.29,24.87,98.83,8.20,5.54,5.68,5.05,7.24,122.07,14.11,7.36,82.21,0,0,19.32,49.71,22.90,53.90
2,3,Netherlands,Developed,14.70,80.43,107.41,17,Female,Urban,Low,Smartphone,6.10,School,Law,3.66,5.45,4.43,10.86,52.52,Sometimes,0.35,1.44,2.65,0.62,99.80,5.02,8.05,1,29.85,Medium,47.74,19.88,84.45,5.76,5.97,5.30,4.16,6.21,133.58,18.37,6.72,26.59,0,0,15.68,51.47,38.87,51.88
3,4,Argentina,Developing,18.99,68.14,69.08,18,Female,Rural,Middle,Smartphone,5.98,Diploma,Social Sciences,6.35,7.62,4.32,12.39,46.99,Always,0.36,1.44,2.52,0.37,53.76,13.42,4.52,3,28.24,Medium,49.79,24.18,90.11,9.20,5.40,6.19,7.48,4.60,133.29,17.05,7.78,97.83,0,1,20.24,59.18,31.66,51.14
4,5,Vietnam,Developing,30.70,66.87,45.63,21,Female,Rural,Low,Shared Device,6.70,Graduate,Business,7.80,7.74,3.11,8.87,40.93,Never,0.65,1.23,1.23,0.39,81.35,7.92,3.82,0,10.99,Low,60.00,28.71,93.98,10.00,6.81,3.38,3.57,2.42,88.52,11.63,5.26,37.78,0,0,8.29,71.06,15.35,46.64


## Summary

- Loaded `df_clean` (100,000 rows) from notebook 02.
- Ran a four-point validation pass: nulls, column naming, dtype correctness, duplicates.
- Generated a per-column data quality report with completeness, range, and dtype.
- Saved the final dataframe to `data/processed/cleaned_data.csv` (index=False).
- Confirmed file size and shape via a read-back check.
- ETL pipeline complete.